# Sionna 2.0 — 915.95 MHz Simulation (Nottingham)

**Frequency:** 915.95 MHz · **RX:** 1200 sequential from CSV · **Terrain:** flat · **Backend:** PyTorch (Sionna 2.0)

**CSV:** nottingham915.csv  ·  TX EIRP=50.3 dBm  ·  RX system gain=−7.8 dB

Adapted from `sionna019_900mhz_simulation.ipynb` using the Sionna 2.0 API from `sionna2_main_simulation.ipynb`.

## Cell 0 — Environment Setup

Sionna 2.0 uses PyTorch — no TensorFlow, no Mitsuba variant setup needed.

In [ ]:
import os, sys, json, csv, time, warnings, glob, re
import xml.etree.ElementTree as ET
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib; matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 100})
import matplotlib.pyplot as plt
from scipy.constants import speed_of_light as C
from pyproj import Transformer
from datetime import datetime
import math

# Sionna 2.0 — PyTorch backend, no TensorFlow, no Mitsuba variant setup
import torch
import sionna
import sionna.rt as rt
from sionna.rt import load_scene, RadioMaterial, PlanarArray, Transmitter, Receiver, PathSolver

_HAS_RIO = False
try:
    import rasterio as rio
    _HAS_RIO = True
    print('rasterio: OK')
except ImportError:
    print('rasterio: NOT available — flat terrain fallback will be used')

_HAS_OSM = False
try:
    import osmnx as ox
    _HAS_OSM = True
    print('osmnx   : OK')
except ImportError:
    print('osmnx   : NOT available')

def _safe(v):
    """Convert tensor or numeric value to Python float safely."""
    if hasattr(v, 'item'):  return float(v.item())
    if hasattr(v, 'numpy'): return float(v.numpy())
    return float(v)

print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'Sionna  : {sionna.__version__}')

## Cell 1 — Configuration (900 MHz / Nottingham)

In [ ]:
# ── City / scene ─────────────────────────────────────────────────────────────
CITY_NAME    = 'Nottingham'
UTM_EPSG     = 32630          # UTM zone 30N (UK)

# ── Scene bbox (WGS84) ────────────────────────────────────────────────────────
SCENE_WEST   = -1.260093
SCENE_EAST   = -1.129307
SCENE_SOUTH  =  52.945798
SCENE_NORTH  =  52.998702

# ── TX parameters ─────────────────────────────────────────────────────────────
TX_LON              = -1.2559
TX_LAT              =  52.9863
TX_AGL_M            = 17.0           # m  (CSV: Tx antenna height = 17 m)
TX_CONDUCTED_DBM    = 49.0           # dBm (50.3 amp − 1.3 cable loss)
TX_ANTENNA_GAIN_DBI =  1.3           # dBi (collinear omni)
EIRP_DBM            = TX_CONDUCTED_DBM + TX_ANTENNA_GAIN_DBI  # 50.3 dBm

# ── Antenna pattern ───────────────────────────────────────────────────────────
ANTENNA_PATTERN = 'iso'   # 'donut' = half-wave dipole shape | 'iso' = isotropic

# ── RX parameters ─────────────────────────────────────────────────────────────
RX_AGL_M           =  1.5            # m  (CSV: Rx antenna height = 1.5 m)
RX_EXTRA_GAIN_DB   = -7.8            # dB (system gain: antenna + cable + filters)
SITE_CORRECTION_DB =  0.0            # dB (calibration offset)
NOISE_FLOOR_DBM    = -124.0          # dBm (CSV: System noise floor)

# ── RX selection ──────────────────────────────────────────────────────────────
NUM_RX       = 1200              # first 1200 sequential rows from CSV

# ── Frequency ─────────────────────────────────────────────────────────────────
FREQUENCY_HZ = 915.95e6          # Hz  (CSV: Frequency = 915.95 MHz)

# ── Terrain ───────────────────────────────────────────────────────────────────
FLAT_TERRAIN = True              # True = flat z=0; False = use DEM GeoTIFF

# ── Ray tracing ───────────────────────────────────────────────────────────────
MAX_DEPTH        = 8             # bounces
NUM_SAMPLES_PS   = 1_000_000     # rays per batch
BATCH_SIZE       = 5             # RX per compute_paths() call

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR        = '/home/georgeskai/Documents/FYP2026/nottingham900'
SCENE_DIR       = os.path.join(BASE_DIR, 'scene')
BLENDER_DIR     = '/home/georgeskai/Documents/FYP2026/nottingham900/Nottingham'
SCENE_XML       = os.path.join(BLENDER_DIR, 'scene_sionna2.xml')  # Blender → Sionna 2.0 converted scene
OUT_DIR         = os.path.join(BASE_DIR, 'results')
os.makedirs(OUT_DIR, exist_ok=True)
DEM_TIFF        = os.path.join(BASE_DIR, 'uk_terrain_nottingham_aoi.tif')

OFCOM_RAW_CSV   = '/home/georgeskai/Documents/FYP2026/nottingham900/nottingham915.csv'
RX_CSV          = os.path.join(SCENE_DIR, 'receiver_locations.csv')
MEASUREMENT_CSV = os.path.join(SCENE_DIR, 'measurements_with_pathloss.csv')

# ── RSSI from path gain (Sionna 2.0) ─────────────────────────────────────────
def rssi_from_path_gain(path_gain_linear):
    """Convert linear path gain |a|² to RSSI in dBm.
    paths.a already includes TX+RX antenna gains in Sionna 2.0.
    Use TX_CONDUCTED (not EIRP) to avoid double-counting antenna gain.
    """
    pg = np.asarray(path_gain_linear, dtype=float)
    pl = np.where(pg > 0, -10.0 * np.log10(np.maximum(pg, 1e-30)), np.nan)
    return TX_CONDUCTED_DBM - pl + RX_EXTRA_GAIN_DB + SITE_CORRECTION_DB

print(f'Frequency        : {FREQUENCY_HZ/1e6:.2f} MHz')
print(f'TX               : lon={TX_LON}  lat={TX_LAT}  AGL={TX_AGL_M}m')
print(f'TX conducted     : {TX_CONDUCTED_DBM} dBm   antenna={TX_ANTENNA_GAIN_DBI} dBi   EIRP={EIRP_DBM:.1f} dBm')
print(f'Antenna pattern  : {ANTENNA_PATTERN}')
print(f'RX system gain   : {RX_EXTRA_GAIN_DB} dB')
print(f'Noise floor      : {NOISE_FLOOR_DBM} dBm')
print(f'NUM_RX           : {NUM_RX}')
print(f'MAX_DEPTH        : {MAX_DEPTH}   NUM_SAMPLES_PS : {NUM_SAMPLES_PS:,}   BATCH_SIZE : {BATCH_SIZE}')
print(f'BASE_DIR         : {BASE_DIR}')
print(f'SCENE_XML        : {SCENE_XML}')
print(f'OFCOM_RAW_CSV    : {OFCOM_RAW_CSV}')

## GPS → Scene Coordinate Transform — Method & Reference

All geographic coordinates in this notebook are converted to Sionna scene-local metres using a two-stage pipeline consistent with the Sionna RT community approach (geo2sigmap, sionna-large-radio-maps).

**Stage 1 — WGS84 to UTM.** The raw GPS coordinates (EPSG:4326, decimal degrees) are projected to UTM Zone 30N (EPSG:32630) using pyproj `Transformer.from_crs` with `always_xy=True`. The `always_xy` flag enforces `(longitude, latitude)` → `(easting, northing)` axis order regardless of the CRS convention, preventing the silent axis swap that is a common source of multi-kilometre placement errors.

**Stage 2 — UTM to scene-local.** The scene origin is defined as the arithmetic midpoint of the scene bounding box `((SCENE_WEST+SCENE_EAST)/2, (SCENE_SOUTH+SCENE_NORTH)/2)`. Every antenna position is expressed as the signed metre offset from this origin `(easting − origin_easting, northing − origin_northing)`. This matches the origin used by the OSM scene builder.

**Stage 3 — Terrain height.** Antenna height is expressed as height Above Ground Level (AGL). Ground elevation at each antenna site is sampled by bilinear interpolation from a merged SRTM/LiDAR DEM (GeoTIFF, WGS84, 1–30 m resolution). When the DEM file is absent the notebook falls back to flat terrain (z=0 everywhere).

**Reference:** pyproj `Transformer` (RFC 7946, EPSG:4326 ↔ EPSG:32630); Sionna RT positions are pure Cartesian metres — confirmed in NVlabs/sionna Discussion #154.

## Cell 2 — Coordinate Utilities (GPS → UTM → Local, DEM Terrain)

In [ ]:
# ── Coordinate transformers (pyproj, WGS84 ↔ UTM 30N) ──────────────────────
# always_xy=True enforces (lon, lat) / (easting, northing) order
gps_to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
utm_to_gps = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)

# Scene centre in UTM (origin of local coordinate system)
center_lon = (SCENE_WEST + SCENE_EAST)  / 2
center_lat = (SCENE_SOUTH + SCENE_NORTH) / 2
utm_center_x, utm_center_y = gps_to_utm.transform(center_lon, center_lat)

def gps_to_local(lon, lat, height=0.0):
    """WGS84 (lon, lat) → scene-local (x, y, z) in metres.
    Origin = scene bbox centre. X = east, Y = north, Z = up."""
    ux, uy = gps_to_utm.transform(lon, lat)
    return float(ux - utm_center_x), float(uy - utm_center_y), float(height)

def local_to_gps(x, y):
    """Scene-local (x, y) → WGS84 (lon, lat)."""
    lon, lat = utm_to_gps.transform(x + utm_center_x, y + utm_center_y)
    return float(lon), float(lat)

# ── DEM terrain elevation ─────────────────────────────────────────────────────
dem_data = dem_nodata = dem_tf = dem_crs = None
if _HAS_RIO and not FLAT_TERRAIN and os.path.exists(DEM_TIFF):
    _src      = rio.open(DEM_TIFF)
    dem_data  = _src.read(1).astype(np.float32)
    dem_nodata= _src.nodata
    dem_tf    = _src.transform
    dem_crs   = str(_src.crs)
    print(f'DEM loaded : {dem_data.shape}  nodata={dem_nodata}  CRS={dem_crs}')
else:
    if FLAT_TERRAIN:
        print('DEM        : FLAT_TERRAIN=True — z=0 everywhere')
    else:
        print(f'DEM        : not found at {DEM_TIFF} — falling back to flat terrain')

def get_dem_elevation(local_x, local_y):
    """Return absolute terrain elevation (m ASL) at scene-local (x, y)."""
    if dem_data is None: return 0.0
    utm_x = float(local_x) + utm_center_x
    utm_y = float(local_y) + utm_center_y
    # DEM is in WGS84 lon/lat — convert UTM → lon/lat for pixel lookup
    lon, lat = utm_to_gps.transform(utm_x, utm_y)
    col_f, row_f = ~dem_tf * (lon, lat)
    r, c = int(np.floor(row_f)), int(np.floor(col_f))
    H, W = dem_data.shape
    if 0 <= r < H-1 and 0 <= c < W-1:
        dr, dc = row_f - r, col_f - c
        z = ((1-dr)*(1-dc)*dem_data[r,c]   + (1-dr)*dc*dem_data[r,c+1] +
              dr*(1-dc)*dem_data[r+1,c]    + dr*dc*dem_data[r+1,c+1])
        if dem_nodata is None or not np.isclose(float(z), dem_nodata):
            return float(z)
    return 0.0

# Scene z=0 corresponds to DEM elevation at scene centre.
_SCENE_ORIGIN_ELEV = get_dem_elevation(0.0, 0.0)

def terrain_z(local_x, local_y):
    """Return scene-local Z of ground at (x,y). 0.0 if flat terrain."""
    if FLAT_TERRAIN or dem_data is None: return 0.0
    return get_dem_elevation(local_x, local_y) - _SCENE_ORIGIN_ELEV

print(f'Scene centre  : lon={center_lon:.6f}  lat={center_lat:.6f}')
print(f'UTM centre    : ({utm_center_x:.1f}, {utm_center_y:.1f})')
print(f'Scene origin elevation: {_SCENE_ORIGIN_ELEV:.1f} m ASL')
print(f'gps_to_local(TX): {gps_to_local(TX_LON, TX_LAT)}')

## Cell 3 — Load Scene (Sionna 2.0)

In [ ]:
import gc, os
gc.collect()

if not os.path.exists(SCENE_XML):
    raise FileNotFoundError(
        f'Scene not found: {SCENE_XML}\n'
        'Run CELL B2 in scene builder to generate scene_sionna2.xml first.')

# ── Patch Sionna ITU validator to allow extrapolation below 1 GHz ────────────
# At 915 MHz Sionna raises ValueError for all ITU materials (valid ≥1 GHz).
# Monkeypatch clamps f to valid minimum so model extrapolates instead of raises.
import importlib as _il
_PATCHED_ITU = False
try:
    # Patch the reference inside itu_material.py — that's where cb() calls it from
    _itu_mat_mod = _il.import_module('sionna.rt.radio_materials.itu_material')
    _itu_fn_mod  = _il.import_module('sionna.rt.radio_materials.itu')
    _orig_itu_fn = _itu_mat_mod.itu_material  # the name used by cb() closure

    def _patched_itu(name, f):
        try:
            return _orig_itu_fn(name, f)
        except ValueError:
            return _orig_itu_fn(name, 1.0e9)  # clamp to 1 GHz minimum

    _itu_mat_mod.itu_material = _patched_itu  # patch where cb() looks it up
    _PATCHED_ITU = True
    print('ITU validator patched — sub-1GHz extrapolation enabled')
except Exception as _pe:
    print(f'Could not patch ITU validator: {_pe}')

print(f'Loading scene from {SCENE_XML} ...')
try:
    scene = load_scene(SCENE_XML)
    scene.frequency       = FREQUENCY_HZ   # must be inside patch scope
    scene.synthetic_array = False
finally:
    if _PATCHED_ITU:
        _itu_mat_mod.itu_material = _orig_itu_fn
        print('ITU validator restored.')

# ── Antenna patterns (Sionna 2.0 / PyTorch) ──────────────────────────────────
import numpy as _np
_D_HW = 1.6409

def _tx_pattern_iso(theta, phi):
    _iso = (torch.ones_like(theta) / _np.float32(_np.sqrt(4.0 * _np.pi))).to(torch.complex64)
    return _iso, torch.zeros_like(_iso)

def _tx_pattern_donut(theta, phi):
    _scale = _np.float32((10 ** (TX_ANTENNA_GAIN_DBI / 10) / _D_HW) ** 0.5)
    cos_t  = torch.cos(theta)
    sin_t  = torch.sin(theta)
    safe_s = torch.where(torch.abs(sin_t) < 1e-6, torch.ones_like(sin_t)*1e-6, sin_t)
    f_theta = (_scale * torch.cos(torch.tensor(_np.pi/2.0)*cos_t)/safe_s).to(torch.complex64)
    return f_theta, torch.zeros_like(f_theta)

def _make_array(pattern_fn, fallback='iso'):
    try:
        arr = PlanarArray(num_rows=1, num_cols=1, vertical_spacing=0.5,
                          horizontal_spacing=0.5, pattern=pattern_fn, polarization='V')
        return arr, 'callable'
    except Exception as _e:
        arr = PlanarArray(num_rows=1, num_cols=1, vertical_spacing=0.5,
                          horizontal_spacing=0.5, pattern=fallback, polarization='V')
        return arr, f'fallback={fallback}'

tx_pattern = _tx_pattern_donut if ANTENNA_PATTERN == 'donut' else _tx_pattern_iso
scene.tx_array, _tx_mode = _make_array(tx_pattern)
scene.rx_array, _rx_mode = _make_array(_tx_pattern_iso)

print(f'Scene loaded   : {len(scene.objects)} objects')
print(f'Frequency      : {FREQUENCY_HZ/1e6:.2f} MHz')
print(f'TX antenna     : {ANTENNA_PATTERN}  [{_tx_mode}]')
print(f'Materials      : {len(scene.radio_materials)} in scene')


## Cell 4 — Place TX

In [ ]:
# Remove previous TX
for _n in list(scene.transmitters.keys()):
    scene.remove(_n)

tx_x, tx_y, _ = gps_to_local(TX_LON, TX_LAT)
tx_z = terrain_z(tx_x, tx_y) + TX_AGL_M  # ground + AGL

tx = Transmitter(name='tx0',
                 position=[tx_x, tx_y, tx_z],
                 orientation=[0.0, 0.0, 0.0])
scene.add(tx)

print(f'TX placed:')
print(f'  GPS      : lon={TX_LON}  lat={TX_LAT}')
print(f'  Local    : ({tx_x:.1f}, {tx_y:.1f}, {tx_z:.1f}) m')
print(f'  AGL      : {TX_AGL_M} m  (terrain_z={tx_z - TX_AGL_M:.1f} m)')

## Cell 5 — Extract RX from Ofcom CSV

In [ ]:
import csv as _csv_mod

print('=' * 60)
print('CELL 5 — RX EXTRACTION (first 1200 sequential from CSV)')
print('=' * 60)

if not os.path.exists(OFCOM_RAW_CSV):
    raise FileNotFoundError(f'CSV not found: {OFCOM_RAW_CSV}')

# ── Auto-detect header row ────────────────────────────────────────────────────
# Scan up to the first 40 lines for the row that contains both
# 'Latitude' and 'Longitude' — works regardless of metadata lines before data
_hdr_idx = None
with open(OFCOM_RAW_CSV, 'r', encoding='utf-8', errors='replace') as _f:
    for _i, _line in enumerate(_f):
        if 'Latitude' in _line and 'Longitude' in _line:
            _hdr_idx = _i
            break
        if _i > 40:
            break

if _hdr_idx is None:
    raise ValueError(
        f'Could not find header row in {OFCOM_RAW_CSV}\n'
        'Expected a line containing both "Latitude" and "Longitude" in the first 40 lines.')

print(f'Header at line {_hdr_idx + 1}  (0-based index {_hdr_idx})')
_df_raw = pd.read_csv(OFCOM_RAW_CSV, skiprows=_hdr_idx, low_memory=False)
print(f'Columns: {list(_df_raw.columns)}')
print(f'Total rows: {len(_df_raw)}')

# ── Column mapping — flexible: match by keyword ───────────────────────────────
def _find_col(df, *keywords):
    """Return first column name that contains all keywords (case-insensitive)."""
    for col in df.columns:
        c = col.strip().lower()
        if all(k.lower() in c for k in keywords):
            return col
    return None

_lat_col  = _find_col(_df_raw, 'latitude')
_lon_col  = _find_col(_df_raw, 'longitude')
_rssi_col = _find_col(_df_raw, 'measurement') or _find_col(_df_raw, 'dBm') or _find_col(_df_raw, 'dbm')

if not _lat_col:
    raise KeyError(f'No Latitude column found. Available: {list(_df_raw.columns)}')
if not _lon_col:
    raise KeyError(f'No Longitude column found. Available: {list(_df_raw.columns)}')
if not _rssi_col:
    raise KeyError(f'No RSSI/measurement column found. Available: {list(_df_raw.columns)}')

print(f'Lat  col : {_lat_col!r}')
print(f'Lon  col : {_lon_col!r}')
print(f'RSSI col : {_rssi_col!r}')

# Drop rows with non-numeric values in key columns
for _c in [_lat_col, _lon_col, _rssi_col]:
    _df_raw[_c] = pd.to_numeric(_df_raw[_c], errors='coerce')
_df_raw = _df_raw.dropna(subset=[_lat_col, _lon_col, _rssi_col]).reset_index(drop=True)
print(f'Rows after numeric filter: {len(_df_raw)}')

# Take first NUM_RX rows in CSV order (sequential as recorded)
_sel = _df_raw.head(NUM_RX).copy()

# Distance from TX (for info only)
_dlon_m = 111000.0 * math.cos(math.radians(TX_LAT))
_dlat_m = 111000.0
_sel['_dist_km'] = (
    ((_sel[_lat_col] - TX_LAT) * _dlat_m)**2 +
    ((_sel[_lon_col] - TX_LON) * _dlon_m)**2
)**0.5 / 1000.0

print(f'\nSelected  : {len(_sel)} receivers (rows 1–{len(_sel)}, sequential CSV order)')
print(f'Dist range: {_sel["_dist_km"].min():.3f} – {_sel["_dist_km"].max():.3f} km')
print(f'RSSI range: {_sel[_rssi_col].min():.1f} – {_sel[_rssi_col].max():.1f} dBm')

# ── Write receiver_locations.csv ──────────────────────────────────────────────
os.makedirs(os.path.dirname(RX_CSV), exist_ok=True)
with open(RX_CSV, 'w', newline='') as _f:
    _w = _csv_mod.writer(_f)
    _w.writerow(['name', 'lon', 'lat', 'height'])
    for _idx, _row in _sel.iterrows():
        _w.writerow([f'RX_{_idx:06d}',
                     f'{float(_row[_lon_col]):.6f}',
                     f'{float(_row[_lat_col]):.6f}',
                     RX_AGL_M])
print(f'\nWritten : {RX_CSV}  ({len(_sel)} rows)')

# ── Write measurements_with_pathloss.csv ──────────────────────────────────────
with open(MEASUREMENT_CSV, 'w', newline='') as _f:
    _w = _csv_mod.writer(_f)
    _w.writerow(['name', 'lon', 'lat', 'local_measurement_dBm', 'path_loss_dB'])
    for _idx, _row in _sel.iterrows():
        _rssi = float(_row[_rssi_col])
        _pl   = EIRP_DBM - _rssi + RX_EXTRA_GAIN_DB
        _w.writerow([f'RX_{_idx:06d}',
                     f'{float(_row[_lon_col]):.6f}',
                     f'{float(_row[_lat_col]):.6f}',
                     f'{_rssi:.2f}',
                     f'{_pl:.2f}'])
print(f'Written : {MEASUREMENT_CSV}  ({len(_sel)} rows)')

## Cell 6 — Place Receivers

In [ ]:
df_rx = pd.read_csv(RX_CSV)
for nm in list(scene.receivers.keys()):
    scene.remove(nm)

receivers = []
for _, row in df_rx.iterrows():
    lx, ly, _ = gps_to_local(float(row['lon']), float(row['lat']))
    lz = terrain_z(lx, ly) + RX_AGL_M  # ground + AGL
    rx = Receiver(name=row['name'], position=[lx, ly, lz])
    scene.add(rx)
    receivers.append(rx)

print(f'Placed {len(receivers)} receivers at z={RX_AGL_M}m (flat terrain={FLAT_TERRAIN})')

## Cell A — paths.a Normalization Diagnostic

Compares `sum(|paths.a|²)` to theoretical FSPL at 3 distances. Reveals if Sionna 2.0 PathSolver has different amplitude normalization than expected. Run after Cell 6 (receivers placed), before DIAG.

In [ ]:
# ====================================================================
# CELL A — paths.a NORMALIZATION DIAGNOSTIC
# ====================================================================
# Runs a pure LOS path solve at 3 known distances, prints paths.a.shape
# and compares sum(|a|²) to FSPL — reveals any normalization offset.
# Run after Cell 4 (TX placed) and Cell 6 (RX list loaded).
# ====================================================================
import numpy as np, math, time
print("=" * 70)
print("CELL A — paths.a NORMALIZATION CHECK (LOS comparison to FSPL)")
print("=" * 70)

C = 3e8
_lam = C / FREQUENCY_HZ  # wavelength

def _fspl_linear(d):
    """Free-space path gain (linear) = (λ/4πr)²"""
    return (_lam / (4 * math.pi * d)) ** 2

def _fspl_db(d):
    return -10 * math.log10(_fspl_linear(d))

tx_obj = list(scene.transmitters.values())[0]
_tx_lx = _safe(tx_obj.position[0])
_tx_ly = _safe(tx_obj.position[1])
_tx_lz = _safe(tx_obj.position[2])

print(f"\nWavelength     : {_lam:.4f} m")
print(f"TX position    : ({_tx_lx:.1f}, {_tx_ly:.1f}, {_tx_lz:.1f}) m")
print(f"RX AGL         : {RX_AGL_M} m")
print()
print(f"  {'Dist':>6}  {'FSPL(dB)':>9}  {'a.shape(raw)':>22}  {'sum|a|²(dB)':>12}  {'vs FSPL':>9}  {'paths':>6}  {'time(s)':>7}")
print(f"  {'-'*90}")

_test_dists = [50, 200, 500, 1000, 2000]

for _test_d in _test_dists:
    _rx_lx = _tx_lx + _test_d  # due East
    _rx_lz = RX_AGL_M
    _rx_test = Receiver(name='_debug_rx', position=[_rx_lx, _tx_ly, _rx_lz])
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    scene.add(_rx_test)

    _t0 = time.time()
    try:
        _paths = PathSolver()(scene,
            max_depth=MAX_DEPTH,
            los=True,
            specular_reflection=True,
            diffraction=True,
            edge_diffraction=True,
            diffuse_reflection=True,
            samples_per_src=2_000_000)
        _dt = time.time() - _t0

        # Combine real/imag tuple → complex array
        _a_raw = _paths.a
        if isinstance(_a_raw, tuple):
            _raw_shape = ('tuple', tuple(_a_raw[0].shape))
            _a_np = ((_a_raw[0].numpy() if hasattr(_a_raw[0], 'numpy') else np.array(_a_raw[0])) +
                     1j*(_a_raw[1].numpy() if hasattr(_a_raw[1], 'numpy') else np.array(_a_raw[1])))
        else:
            _raw_shape = tuple(_a_raw.shape)
            _a_np = _a_raw.numpy() if hasattr(_a_raw, 'numpy') else np.array(_a_raw)

        # Squeeze
        _a_sq = np.squeeze(_a_np)

        # Print shapes for debug
        _sum_pwr = float(np.sum(np.abs(_a_sq) ** 2))
        _n_valid = int(np.sum(np.abs(_a_sq) ** 2 > 1e-30))
        if _sum_pwr > 1e-30:
            _sum_db = 10 * math.log10(_sum_pwr)
        else:
            _sum_db = float('nan')

        _fspl = _fspl_db(_test_d)
        _vs_fspl = _sum_db - (-_fspl)  # sum_db is negative, FSPL is positive loss
        # correct comparison: sum|a|² (dB) should ≈ -FSPL (negative)
        _vs_fspl2 = _sum_db - (-_fspl)

        print(f"  {_test_d:>6}m  {_fspl:>9.1f}  {str(_raw_shape):>22}  {_sum_db:>12.2f}  {(_sum_db + _fspl):>+9.1f}  {_n_valid:>6}  {_dt:>7.1f}")

    except Exception as _e:
        _dt = time.time() - _t0
        print(f"  {_test_d:>6}m  ERROR: {_e}  ({_dt:.1f}s)")

print()
print("  Expected: sum|a|²(dB) ≈ -(FSPL dB) for open LOS.")
print("  'vs FSPL' = sum|a|²(dB) + FSPL(dB)  (should be ~0 to +10 dB for urban overhead)")
print()
print("  paths.a raw shape legend:")
print("  Sionna 2.0 typical: [num_rx, num_tx, num_rx_ant, num_tx_ant, num_paths]")
print("  or: [batch, num_rx, ...] — check documentation for your version")
print()

# Restore receivers
for _n in list(scene.receivers.keys()): scene.remove(_n)
_rxlist_debug = list(receivers) if 'receivers' in dir() else []
for _rx in _rxlist_debug: scene.add(_rx)
print(f"  Receivers restored: {len(_rxlist_debug)}")
print()
print("  ── Top 5 paths for last test distance ──────────────────────────────")
try:
    _rx_test2 = Receiver(name='_debug_rx2', position=[_tx_lx + 200, _tx_ly, RX_AGL_M])
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    scene.add(_rx_test2)
    _paths2 = PathSolver()(scene,
        max_depth=MAX_DEPTH, los=True, specular_reflection=True,
        diffraction=True, edge_diffraction=True, diffuse_reflection=True,
        samples_per_src=2_000_000)
    _a2_raw = _paths2.a
    if isinstance(_a2_raw, tuple):
        _a2 = np.squeeze(
            (_a2_raw[0].numpy() if hasattr(_a2_raw[0], 'numpy') else np.array(_a2_raw[0])) +
            1j*(_a2_raw[1].numpy() if hasattr(_a2_raw[1], 'numpy') else np.array(_a2_raw[1])))
        print(f"  paths.a: tuple(real,imag), real shape={_a2_raw[0].shape}")
    else:
        _a2 = np.squeeze(_a2_raw.numpy() if hasattr(_a2_raw, 'numpy') else np.array(_a2_raw))
    print(f"  paths.a after squeeze shape: {_a2.shape}  dtype: {_a2.dtype}")
    _pwr2 = np.abs(_a2.flatten()) ** 2
    _ord2 = np.argsort(_pwr2)[::-1]
    for _ri in range(min(5, len(_ord2))):
        _pi = _ord2[_ri]
        _pw = _pwr2[_pi]
        if _pw > 1e-40:
            print(f"    rank {_ri+1}: |a|²={_pw:.4e}  ({10*math.log10(_pw):.1f} dB)")
    print(f"  Expected LOS |a|² at 200m = {_fspl_linear(200):.4e}  ({-_fspl_db(200):.1f} dB)")
    # Friis check
    _rssi_check = TX_CONDUCTED_DBM + 10*math.log10(max(_pwr2[_ord2[0]], 1e-40)) + RX_EXTRA_GAIN_DB
    print(f"  RSSI from strongest path: {_rssi_check:.1f} dBm  (expected ~{TX_CONDUCTED_DBM - _fspl_db(200) + RX_EXTRA_GAIN_DB:.1f} dBm for FSPL)")
    # Tau
    if hasattr(_paths2, 'tau'):
        _tau2 = np.squeeze(np.array(_paths2.tau))
        print(f"  paths.tau shape: {_tau2.shape}")
        _tau_flat = _tau2.flatten()
        _valid_tau = _tau_flat[_tau_flat > 0]
        if len(_valid_tau):
            _los_tau = 200 / C
            print(f"  Expected LOS tau: {_los_tau:.6f} s  ({200}m/c)")
            print(f"  Min tau found: {_valid_tau.min():.6f} s  ({_valid_tau.min()*C:.1f}m)")
except Exception as _e2:
    print(f"  ERROR: {_e2}")
    import traceback; traceback.print_exc()
finally:
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    for _rx in _rxlist_debug: scene.add(_rx)
    print(f"  Receivers restored: {len(_rxlist_debug)}")


## Cell DIAG — Step-by-Step Bias Diagnostic

Run **before the path solver** to verify TX/RX positions, antenna heights, and scene geometry.
Tests 10 → 100 receivers to catch systematic bias early. Uses Sionna 2.0 `compute_paths()` API.

In [ ]:
from sionna.rt import PathSolver
# ====================================================================
# CELL DIAG — Step-by-Step Bias Diagnostic (10 → 100 receivers)
# ====================================================================
# Tests RSSI formula, RX heights, TX position, and geometry systematically.
# Run BEFORE CELL 9b to isolate the source of high RMSE.
# ====================================================================
import numpy as np, pandas as pd, math, os, time
from pyproj import Transformer as _Tr

print("=" * 70)
print("BIAS DIAGNOSTIC — Step-by-step RMSE decomposition")
print("=" * 70)

# ── Load measurements ─────────────────────────────────────────────────────────
_df_meas = pd.read_csv(MEASUREMENT_CSV)
print(f"\nMeasurements loaded: {len(_df_meas)} rows")
print(f"  RSSI range   : {_df_meas['local_measurement_dBm'].min():.1f} → {_df_meas['local_measurement_dBm'].max():.1f} dBm")
print(f"  PL range     : {_df_meas['path_loss_dB'].min():.1f} → {_df_meas['path_loss_dB'].max():.1f} dB")

# ── STEP 1: Formula check — FSPL vs measured at known distances ────────────────
print("\n" + "─" * 60)
print("STEP 1 — Free-Space Path Loss formula validation")
print("─" * 60)
C = 3e8
_f = FREQUENCY_HZ
_fspl_fn = lambda d: 20*np.log10(4*np.pi*d*_f/C)

_tx_lon, _tx_lat = TX_LON, TX_LAT
_gps2utm = Transformer.from_crs("EPSG:4326", f"EPSG:{UTM_EPSG}", always_xy=True)
_tx_x, _tx_y = _gps2utm.transform(_tx_lon, _tx_lat)

_rx_x = np.array([_gps2utm.transform(r['lon'], r['lat'])[0] for _, r in _df_meas.iterrows()])
_rx_y = np.array([_gps2utm.transform(r['lon'], r['lat'])[1] for _, r in _df_meas.iterrows()])
_dist = np.sqrt((_rx_x - _tx_x)**2 + (_rx_y - _tx_y)**2)
_df_meas = _df_meas.copy()
_df_meas['dist_m'] = _dist

# Near receivers (50-300m) — most likely LOS → compare against FSPL
_near = _df_meas[_df_meas['dist_m'].between(50, 300)].copy()
_near['fspl_db']    = _near['dist_m'].apply(_fspl_fn)
_near['measured_pl'] = _near['path_loss_dB']
_near['vs_fspl']    = _near['measured_pl'] - _near['fspl_db']
print(f"  Near receivers (50-300m): {len(_near)}")
print(f"  TX_CONDUCTED={TX_CONDUCTED_DBM:.1f} dBm  RX_EXTRA={RX_EXTRA_GAIN_DB:.1f} dB  SITE_CORR={SITE_CORRECTION_DB:.1f} dB")
print(f"  RSSI formula: RSSI = TX_CONDUCTED - PL + RX_EXTRA + SITE_CORR")
print(f"              = {TX_CONDUCTED_DBM:.1f} - PL + {RX_EXTRA_GAIN_DB:.1f} + {SITE_CORRECTION_DB:.1f}")
print(f"  → PL = {TX_CONDUCTED_DBM + RX_EXTRA_GAIN_DB + SITE_CORRECTION_DB:.1f} - RSSI  (= rssi_from_path_gain inverse)")
print()
print(f"  {'Name':<12} {'Dist(m)':>8} {'RSSI(dBm)':>10} {'PL_meas(dB)':>12} {'FSPL(dB)':>9} {'PL-FSPL(dB)':>12}")
for _, r in _near.head(10).iterrows():
    print(f"  {str(r['name']):<12} {r['dist_m']:>8.0f} {r['local_measurement_dBm']:>10.1f} "
          f"{r['measured_pl']:>12.1f} {r['fspl_db']:>9.1f} {r['vs_fspl']:>12.1f}")
_near_overhead = _near['vs_fspl'].mean()
print(f"\n  Mean PL-FSPL (near): {_near_overhead:+.1f} dB  "
      f"(expected +5 to +15 dB for urban LOS overhead)")
if _near_overhead < 0:
    print(f"  ⚠ Negative overhead → TX_CONDUCTED+RX_EXTRA+SITE_CORR over-estimated")
elif _near_overhead > 25:
    print(f"  ⚠ Very high overhead → TX power under-estimated or RX underground")
else:
    print(f"  ✓ Urban overhead looks physically reasonable")

# ── STEP 2: RX height check ────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("STEP 2 — Receiver height sanity check")
print("─" * 60)
_rxlist = list(scene.receivers.values())
if not _rxlist and os.path.exists(RX_CSV):
    _df_rx2 = pd.read_csv(RX_CSV)
    for _, _row2 in _df_rx2.iterrows():
        _x2  = float(_row2['x_m'])
        _y2  = float(_row2['y_m'])
        _lz2 = float(_row2['z_m'])
        _rx2 = Receiver(name=str(_row2['name']), position=[_x2, _y2, _lz2])
        scene.add(_rx2)
    _rxlist = list(scene.receivers.values())
    print(f"  Loaded {len(_rxlist)} receivers from {RX_CSV} (terrain-corrected z)")
_heights = [_safe(rx.position[2]) for rx in _rxlist[:20]]
print(f"  First 20 RX heights (local Z, m):")
for rx, h in zip(_rxlist[:20], _heights):
    flag = " ⚠ UNDERGROUND" if h < -5 else (" ⚠ TOO HIGH (check terrain)" if h > 300 else "")
    print(f"    {rx.name:<12}  z={h:+.2f}m{flag}")
_neg = sum(1 for h in [_safe(rx.position[2]) for rx in _rxlist] if h < 0)
print(f"\n  Total RX underground (z<0): {_neg} / {len(_rxlist)}")

# ── STEP 3: TX position check ─────────────────────────────────────────────────
print("\n" + "─" * 60)
print("STEP 3 — TX position check")
print("─" * 60)
if 'TX_AGL_M' not in dir(): TX_AGL_M = 17.0
_tx = list(scene.transmitters.values())[0]
_tx_lx, _tx_ly, _tx_lz = _safe(_tx.position[0]), _safe(_tx.position[1]), _safe(_tx.position[2])
_tx_glon, _tx_glat = local_to_gps(_tx_lx, _tx_ly)
print(f"  TX local  : ({_tx_lx:.1f}, {_tx_ly:.1f}, {_tx_lz:.1f}) m")
print(f"  TX GPS    : lat={_tx_glat:.6f}  lon={_tx_glon:.6f}")
print(f"  Expected  : lat={TX_LAT:.6f}  lon={TX_LON:.6f}  h={TX_AGL_M:.1f}m")
_lat_err = abs(_tx_glat - TX_LAT) * 111000
_lon_err = abs(_tx_glon - TX_LON) * 111000 * math.cos(math.radians(TX_LAT))
print(f"  Position error: {_lat_err:.1f}m N-S  {_lon_err:.1f}m E-W  Z={_tx_lz:.1f}m (terrain+AGL)")
print(f"  Terrain at TX  : {_tx_lz - TX_AGL_M:.1f}m  AGL={TX_AGL_M:.1f}m  Total={_tx_lz:.1f}m ✓")
print(f"  TX height: AGL={TX_AGL_M:.1f}m  terrain_z={_tx_lz-TX_AGL_M:.1f}m  total={_tx_lz:.1f}m")
if _lat_err > 50 or _lon_err > 50:
    print("  ⚠ TX position error > 50m — check GPS→UTM→local conversion")
else:
    print("  ✓ TX position OK")

# ── STEP 4: Quick path solver on 50 receivers across bands ─────────────────────
print("\n" + "─" * 60)
print("STEP 4 — Path solver: 50 receivers across distance bands")
print("─" * 60)

# ── Scatter correction helpers (mirrors Cell 9b) ──────────────────────────────
_DIAG_SCAT_KEEP = 1.0  # Sionna 2.0 — deterministic, no scat_keep_prob correction

def _diag_extract_types(paths, shape_2d):
    """Return per-path type array (0=LOS,1=refl,2=diff,3=scat) or None."""
    for attr in ('types', 'type', 'interactions', 'interaction_types'):
        t = getattr(paths, attr, None)
        if t is None: continue
        try:
            t_np = t.numpy() if hasattr(t, 'numpy') else np.array(t)
            t_np = np.squeeze(t_np)
            while t_np.ndim > 2: t_np = t_np[..., 0]
            if t_np.ndim == 1: t_np = t_np[np.newaxis, :]
            if t_np.shape == shape_2d: return t_np.astype(np.int8)
        except Exception: pass
    return None

def _diag_summary(a_row, types_row=None):
    """Path gain → incoherent power linear. Sionna 2.0: no scat_keep_prob correction."""
    pwr = np.abs(a_row) ** 2
    valid = pwr > 1e-30
    if not np.any(valid): return None, 0
    pv = pwr[valid].copy(); av = a_row[valid].copy()
    corr = 1.0 / _DIAG_SCAT_KEEP
    if types_row is not None:
        tv = types_row[valid]; scat_mask = (tv == 3)
        if np.any(scat_mask): pv[scat_mask] *= corr
    else:
        _mx = np.max(pv)
        _heur = pv < (_mx * 1e-3)
        if np.any(_heur): pv[_heur] *= corr
        elif _DIAG_SCAT_KEEP < 1.0: pv *= corr
    return float(np.sum(pv)), int(np.sum(valid))

# ── Distance bands ────────────────────────────────────────────────────────────
_gps2utm3 = Transformer.from_crs("EPSG:4326", f"EPSG:{UTM_EPSG}", always_xy=True)
_df_meas2 = _df_meas.copy()
_xy = _df_meas2.apply(lambda r: _gps2utm3.transform(float(r['lon']), float(r['lat'])), axis=1)
_df_meas2['_lx'] = [xy[0] - utm_center_x for xy in _xy]
_df_meas2['_ly'] = [xy[1] - utm_center_y for xy in _xy]
_df_meas2['_dtx'] = np.sqrt((_df_meas2['_lx'] - _tx_lx)**2 + (_df_meas2['_ly'] - _tx_ly)**2)
_df_meas2 = _df_meas2.sort_values('_dtx').reset_index(drop=True)
# Filter out near-TX outliers (< 50m) — GPS errors / mast co-location artefacts
_df_meas2 = _df_meas2[_df_meas2['_dtx'] >= 100].copy()
_bands_sel = [
    _df_meas2[_df_meas2['_dtx'].between( 100,  300)].head(10),
    _df_meas2[_df_meas2['_dtx'].between( 300,  700)].head(10),
    _df_meas2[_df_meas2['_dtx'].between( 700, 1200)].head(10),
    _df_meas2[_df_meas2['_dtx'].between(1200, 2000)].head(10),
    _df_meas2[_df_meas2['_dtx'] > 2000].head(10),
]
_df_test = pd.concat(_bands_sel).drop_duplicates(subset='name').reset_index(drop=True)
print(f"  Testing {len(_df_test)} receivers across 5 distance bands")
print(f"  Formula: rssi_from_path_gain() — TX_CONDUCTED={TX_CONDUCTED_DBM:.1f} RX_EXTRA={RX_EXTRA_GAIN_DB:.1f} SITE_CORR={SITE_CORRECTION_DB:.1f}")
print("  Scatter correction: disabled (Sionna 2.0 deterministic solver)")
print(f"  Samples: 10M per RX  |  max_depth: {MAX_DEPTH}")

_results50 = []
for _, _mrow in _df_test.iterrows():
    _lx = float(_mrow['_lx']); _ly = float(_mrow['_ly']); _d = float(_mrow['_dtx'])
    _rssi_meas = float(_mrow['local_measurement_dBm'])
    _pl_meas   = float(_mrow['path_loss_dB'])
    _rx_name   = str(_mrow['name'])
    # Use position already set by Cell 6c/7 in scene.receivers — these are
    # terrain-corrected and match the positions used in Cell 9b.
    # Recomputing from CSV or DEM risks using a different elevation reference.
    _scene_pos = {rx.name: rx for rx in _rxlist}
    if _rx_name in _scene_pos:
        _rx_z = float(_safe(_scene_pos[_rx_name].position[2]))
        _gz4  = _rx_z - RX_AGL_M
    else:
        # Fallback: DEM lookup (scene-local = ASL - scene_centre_DTM)
        try:
            _ux4, _uy4 = gps_to_utm.transform(float(_mrow["lon"]), float(_mrow["lat"]))
            _gz4 = get_dem_elevation(_ux4 - utm_center_x, _uy4 - utm_center_y)
        except Exception: _gz4 = 0.0
        _rx_z = _gz4 + RX_AGL_M
    print(f'    [{_rx_name}] ground_z={_gz4:.2f}m  rx_z={_rx_z:.2f}m')
    _rx = Receiver(name=_rx_name, position=[_lx, _ly, _rx_z])
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    scene.add(_rx)
    try:
        _t0 = time.time()
        _paths = PathSolver()(scene, 
            max_depth=MAX_DEPTH, los=True, specular_reflection=True,
            diffraction=True, edge_diffraction=True, diffuse_reflection=True,
 samples_per_src=10_000_000)
        _dt = time.time() - _t0
        _a_raw = _paths.a
        if isinstance(_a_raw, tuple):
            _a = (_a_raw[0].numpy() if hasattr(_a_raw[0], 'numpy') else np.array(_a_raw[0])) + \
                 1j*(_a_raw[1].numpy() if hasattr(_a_raw[1], 'numpy') else np.array(_a_raw[1]))
            print(f'      paths.a: tuple(real,imag), each shape {_a_raw[0].shape}')
        else:
            _a = _a_raw.numpy() if hasattr(_a_raw, 'numpy') else np.array(_a_raw)
            print(f'      paths.a raw shape: {_a.shape}  dtype: {_a.dtype}')
        _a = np.squeeze(_a)
        print(f'      paths.a after squeeze: {_a.shape}  dtype: {_a.dtype}')
        # Normalise to 2-D (n_rx, n_paths) — squeeze can produce 0-D or 1-D scalars
        if _a.ndim == 0: _a = _a.reshape(1, 1)
        elif _a.ndim == 1: _a = _a[np.newaxis, :]
        elif _a.ndim > 2: _a = _a.reshape(1, -1)  # flatten extra dims into paths
        _t_all = _diag_extract_types(_paths, _a.shape)
        _t_row = _t_all[0] if (_t_all is not None and _t_all.ndim > 1) else _t_all
        _pg_sum, _n_paths = _diag_summary(_a[0], _t_row)
        if _pg_sum and _pg_sum > 0:
            _rssi_sim = rssi_from_path_gain(_pg_sum)
            _pl_sim   = -10*math.log10(_pg_sum)
        else:
            _rssi_sim = _pl_sim = float('nan')
            _n_paths  = 0
        _fspl_d = 20*math.log10(4*math.pi*_d*FREQUENCY_HZ/C) if _d > 0 else 0
        _results50.append({'name': _rx_name, 'dist_m': _d,
                           'rssi_sim': _rssi_sim, 'rssi_meas': _rssi_meas,
                           'pl_sim': _pl_sim, 'pl_meas': _pl_meas,
                           'fspl': _fspl_d, 'n_paths': _n_paths, 'dt': _dt})
        _err = _rssi_sim - _rssi_meas if not math.isnan(_rssi_sim) else float('nan')
        _ovr = _pl_sim - _fspl_d if not math.isnan(_pl_sim) else float('nan')
        print(f"  {_rx_name:<12} d={_d:5.0f}m  paths={_n_paths:3d}  "
              f"RSSI sim={_rssi_sim:6.1f} meas={_rssi_meas:6.1f}  "
              f"err={_err:+5.1f}  PL-FSPL={_ovr:+5.1f} dB  ({_dt:.1f}s)")
    except Exception as _e:
        print(f"  {_rx_name:<12} ERROR: {_e}")

# Re-add all receivers
for _n in list(scene.receivers.keys()): scene.remove(_n)
for _rx in _rxlist: scene.add(_rx)

_r10_raw = pd.DataFrame(_results50).dropna(subset=['rssi_sim', 'rssi_meas'])
_RSSI_MAX_VALID = -5.0; _DIST_MIN_VALID = 50.0
_r10_clamped = _r10_raw[(_r10_raw['rssi_sim'] <= _RSSI_MAX_VALID) &
                         (_r10_raw['dist_m']   >= _DIST_MIN_VALID)].copy()
_n_removed = len(_r10_raw) - len(_r10_clamped)
if _n_removed:
    print(f"  Clamped {_n_removed} anomalous RX (sim RSSI>{_RSSI_MAX_VALID}dBm or dist<{_DIST_MIN_VALID}m):")
    for _, _rr in _r10_raw[~_r10_raw.index.isin(_r10_clamped.index)].iterrows():
        print(f"    {_rr['name']:<14} d={_rr['dist_m']:4.0f}m  sim={_rr['rssi_sim']:+.1f}dBm  meas={_rr['rssi_meas']:+.1f}dBm  → EXCLUDED")

_r50 = _r10_clamped
if len(_r50):
    _bias = (_r50['rssi_sim'] - _r50['rssi_meas']).mean()
    _rmse = math.sqrt(((_r50['rssi_sim'] - _r50['rssi_meas'])**2).mean())
    print(f"\n  Valid RX summary ({len(_r50)} receivers, dist≥{_DIST_MIN_VALID:.0f}m):")
    print(f"  bias={_bias:+.1f} dB  RMSE={_rmse:.1f} dB")
    print(f"  PL vs FSPL : mean={(_r50['pl_sim']-_r50['fspl']).mean():+.1f} dB  (urban expect +5 to +20 dB)")
    print(f"\n  Distance-band breakdown:")
    print(f"  {'Band':<12} {'N':>4} {'Bias(dB)':>10} {'RMSE(dB)':>10} {'Mean paths':>11}")
    print(f"  {'-'*52}")
    for _bname, _bmin, _bmax in [('<300m',0,300),('300-700m',300,700),
                                   ('700-1200m',700,1200),('1.2-2km',1200,2000),('>2km',2000,9999)]:
        _rb = _r50[(_r50['dist_m']>=_bmin) & (_r50['dist_m']<_bmax)]
        if not len(_rb): continue
        _berr = _rb['rssi_sim'] - _rb['rssi_meas']
        print(f"  {_bname:<12} {len(_rb):>4} {_berr.mean():>+10.1f} "
              f"{float(np.sqrt((_berr**2).mean())):>10.1f} {_rb['n_paths'].mean():>11.0f}")
    if abs(_bias) <= 5 and _rmse <= 10:
        print(f"\n  ✓ Formula and geometry look correct — proceed to Cell 9b")
    elif abs(_bias) > 10:
        print(f"\n  ⚠ Large bias={_bias:+.1f} dB — check TX_CONDUCTED / RX_EXTRA / SITE_CORRECTION")
    else:
        print(f"\n  ⚠ Large scatter RMSE={_rmse:.1f} dB — likely scene geometry gaps (run CELL 3c)")

# ── STEP 5: Distance-band RMSE using ALL scene receivers ─────────────────────
print("\n" + "─" * 60)
print("STEP 5 — Distance-band RSSI vs FSPL reference (all scene receivers)")
print("─" * 60)
try:
    _scene_rx = {nm: rx for nm, rx in scene.receivers.items()}
    _tx_sx = _safe(tx.position[0]); _tx_sy = _safe(tx.position[1])
    _df_band = pd.read_csv(MEASUREMENT_CSV).dropna(subset=['local_measurement_dBm'])
    _rows = []
    for _, _row in _df_band.iterrows():
        _nm = str(_row['name'])
        if _nm not in _scene_rx: continue
        _rx = _scene_rx[_nm]
        _d3 = math.sqrt((_safe(_rx.position[0])-_tx_sx)**2 + (_safe(_rx.position[1])-_tx_sy)**2)
        _rows.append({'name': _nm, 'dist': _d3, 'rssi_meas': float(_row['local_measurement_dBm'])})
    _df_band2 = pd.DataFrame(_rows)
    print(f"  Matched {len(_df_band2)} receivers  |  dist range: {_df_band2['dist'].min():.0f}–{_df_band2['dist'].max():.0f}m")
    print(f"  Reference: rssi_from_path_gain(FSPL_linear)  [free-space upper bound]\n")
    print(f"  {'Band':<12} {'N':>5}  {'Mean dist':>10}  {'Bias(dB)':>10}  {'RMSE(dB)':>10}")
    print(f"  {'-'*56}")
    for (d0, d1), lbl in zip([(0,100),(100,500),(500,1000),(1000,2000),(2000,99999)],
                               ['0–100m','100–500m','500m–1km','1–2km','>2km']):
        _sub = _df_band2[(_df_band2['dist'] >= d0) & (_df_band2['dist'] < d1)]
        if len(_sub) < 2:
            print(f"  {lbl:<12} {len(_sub):>5}  {'—':>10}  {'—':>10}  {'—':>10}"); continue
        _fspl_lin = (3e8 / (4*np.pi*_sub['dist']*FREQUENCY_HZ))**2
        _rssi_fs  = rssi_from_path_gain(_fspl_lin)   # free-space RSSI upper bound
        _bias_b   = (_rssi_fs - _sub['rssi_meas']).mean()
        _rmse_b   = math.sqrt(((_rssi_fs - _sub['rssi_meas'])**2).mean())
        print(f"  {lbl:<12} {len(_sub):>5}  {_sub['dist'].mean():>9.0f}m  {_bias_b:>+9.1f}  {_rmse_b:>9.1f}")
    print(f"\n  NOTE: STEP 5 uses free-space as reference, not ray tracing.")
    print(f"  Run CELL 9b + 9d for full sim-vs-meas RMSE.")
except Exception as _e5:
    print(f"  STEP 5 error: {_e5}")
    import traceback; traceback.print_exc()


## Cell 7 — Path Solver (900 MHz, Sionna 2.0, Batched)

Full path solver with per-ray extraction, adaptive samples, and CSV output.
Uses Sionna 2.0 API — no `scat_keep_prob` parameter.

In [ ]:
# ====================================================================
# CELL 7 — PATH SOLVER WITH PER-RAY EXTRACTION  [900 MHz / Sionna 2.0]
# ====================================================================
# Processes all receivers in batches of BATCH_SIZE using compute_paths().
# Sionna 2.0 API: no scat_keep_prob parameter.
# paths.a is a complex tensor — power per path = |paths.a|²
# ====================================================================
import gc, time, os, math
import numpy as np, pandas as pd
from datetime import datetime

print('=' * 70)
print('CELL 7 — PATH SOLVER  [900 MHz / Sionna 2.0]')
print('=' * 70)

_safe_ps = lambda v: float(v.item()) if hasattr(v, 'item') else float(v)

# ── Configuration ─────────────────────────────────────────────────────────────
SAVE_PER_RAY    = True
MAX_RAYS_PER_RX = 300
MAX_SAMPLES_PS  = 20_000_000   # hard cap for OOM safety

# Sionna 2.0 deterministic solver — no scat_keep_prob
PS_CONFIG_BASE = dict(
    max_depth        = MAX_DEPTH,
    los              = True,
    specular_reflection = True,
    diffraction      = True,
    diffuse_reflection  = True,
    edge_diffraction = True,
    # no scat_keep_prob in Sionna 2.0
)

_tx_ps = list(scene.transmitters.values())[0]
tx_pos = np.array([_safe_ps(_tx_ps.position[0]),
                   _safe_ps(_tx_ps.position[1]),
                   _safe_ps(_tx_ps.position[2])])
_C_local = 3e8

print(f'  Sionna 2.0 deterministic solver — no scat_keep_prob')
print(f'  TX conducted    : {TX_CONDUCTED_DBM:.1f} dBm  |  RX extra: {RX_EXTRA_GAIN_DB:.1f} dB  |  Site corr: {SITE_CORRECTION_DB:.1f} dB')
print(f'  TX position     : ({tx_pos[0]:.1f}, {tx_pos[1]:.1f}, {tx_pos[2]:.1f}) m')
print(f'  Batch size      : {BATCH_SIZE} receivers  |  Base samples: {NUM_SAMPLES_PS:,}')
for k, v in PS_CONFIG_BASE.items():
    print(f'  {k:15s}: {v}')

def adaptive_samples(dist_m):
    if dist_m > 9000:   return min(NUM_SAMPLES_PS * 2, MAX_SAMPLES_PS)
    elif dist_m > 5000: return min(int(NUM_SAMPLES_PS * 1.5), MAX_SAMPLES_PS)
    else:               return NUM_SAMPLES_PS

def extract_amplitudes(paths):
    """Extract complex CIR amplitudes from Sionna 2.0 Paths object.

    In this Sionna 2.0 build paths.a returns a (real, imag) tuple of
    float32 tensors.  Combine them before any further processing.
    Shape after combining: [num_rx, num_paths] complex64.
    """
    def _to_np(t):
        return t.numpy() if hasattr(t, 'numpy') else np.array(t)

    a = getattr(paths, 'a', None)
    if a is not None:
        try:
            if isinstance(a, tuple):
                a_np = _to_np(a[0]) + 1j * _to_np(a[1])
            else:
                a_np = _to_np(a)
            a_np = np.squeeze(a_np)
            if a_np.ndim == 1:
                a_np = a_np[np.newaxis, :]
            elif a_np.ndim > 2:
                a_np = a_np.reshape(a_np.shape[0], -1)
            return a_np.astype(complex)
        except Exception:
            pass
    # Fallback: paths.cir()
    try:
        a_t, _ = paths.cir()
        if isinstance(a_t, tuple):
            a_np = _to_np(a_t[0]) + 1j * _to_np(a_t[1])
        else:
            a_np = _to_np(a_t)
        a_np = np.squeeze(a_np)
        if a_np.ndim == 1:
            a_np = a_np[np.newaxis, :]
        elif a_np.ndim > 2:
            a_np = a_np.reshape(a_np.shape[0], -1)
        return a_np.astype(complex)
    except Exception:
        return np.zeros((1, 1), dtype=complex)

def extract_tau(paths, num_rx, num_paths):
    tau = getattr(paths, 'tau', None)
    if tau is None:
        return np.full((num_rx, num_paths), np.nan, np.float32)
    try:
        t = tau.numpy() if hasattr(tau, 'numpy') else np.array(tau)
        t = np.squeeze(t)
        while t.ndim > 2: t = t[..., 0]
        if t.ndim == 1: t = t[np.newaxis, :]
        return t
    except Exception:
        return np.full((num_rx, num_paths), np.nan, np.float32)

def summary_metrics(a_row):
    """Compute best/incoherent/coherent path loss from complex amplitude row.
    Sionna 2.0 — no scat_keep_prob correction needed.
    """
    pwr   = np.abs(a_row) ** 2
    valid = pwr > 1e-30
    if not np.any(valid):
        return np.nan, np.nan, np.nan, 0
    pv = pwr[valid]
    av = a_row[valid]
    best_pl       = -10 * np.log10(np.max(pv))
    incoherent_pl = -10 * np.log10(np.sum(pv))
    coh_pwr       = np.abs(np.sum(av)) ** 2
    coherent_pl   = -10 * np.log10(coh_pwr) if coh_pwr > 1e-30 else np.nan
    return best_pl, incoherent_pl, coherent_pl, int(np.sum(valid))

def ray_type_heuristic(path_len, los_dist, pwr, max_pwr):
    if los_dist > 0 and abs(path_len - los_dist) / los_dist < 0.01: return 'LOS'
    ratio = pwr / max_pwr if max_pwr > 0 else 0
    excess = path_len - los_dist
    if excess < 50  and ratio > 0.01:  return 'REFLECTION'
    if excess >= 50 and ratio > 0.001: return 'MULTI_REFLECTION'
    if ratio < 0.01:                   return 'DIFFRACTION'
    if ratio < 0.001:                  return 'SCATTERING'
    return 'UNKNOWN'

def run_batch(batch, cfg):
    for nm in list(scene.receivers.keys()): scene.remove(nm)
    for rx in batch: scene.add(rx)
    try:
        paths = PathSolver()(scene, **cfg)
    except Exception as _oom:
        if any(k in str(_oom).lower() for k in ['oom', 'resource exhausted', 'memory']):
            paths = scene.compute_paths(**{**cfg, 'samples_per_src': 500_000})
        else:
            raise
    return paths

# ── Sort by distance ──────────────────────────────────────────────────────────
_all_rx = list(receivers)
total   = len(_all_rx)
tx_pos2d = tx_pos[:2]
_all_rx.sort(key=lambda rx: float(np.linalg.norm(
    [_safe_ps(rx.position[0]) - tx_pos2d[0], _safe_ps(rx.position[1]) - tx_pos2d[1]])))

ts          = datetime.now().strftime('%Y%m%d_%H%M%S')
summary_csv = os.path.join(OUT_DIR, f'path_solver_summary_900s2_{ts}.csv')
per_ray_csv = os.path.join(OUT_DIR, f'path_solver_per_ray_900s2_{ts}.csv') if SAVE_PER_RAY else None

summary_rows = []
per_ray_rows = []
errors = 0
t0 = time.time()

print(f'\nProcessing {total} receivers in batches of {BATCH_SIZE} ...')

for b_start in range(0, total, BATCH_SIZE):
    batch    = _all_rx[b_start : b_start + BATCH_SIZE]
    max_dist = max(float(np.linalg.norm(
        [_safe_ps(rx.position[0]) - tx_pos2d[0], _safe_ps(rx.position[1]) - tx_pos2d[1]]))
        for rx in batch)
    n_samp = adaptive_samples(max_dist)
    cfg    = {**PS_CONFIG_BASE, 'samples_per_src': n_samp}

    paths = None
    batch_paths = 0
    try:
        paths       = run_batch(batch, cfg)
        a_all       = extract_amplitudes(paths)
        batch_paths = int(np.sum(np.abs(a_all) ** 2 > 1e-30))
    except Exception as _e:
        print(f'  [WARN] Batch {b_start}: {_e}')
        paths = None; a_all = None; batch_paths = 0

    if paths is None or batch_paths == 0:
        for rx in batch:
            los_d = float(np.linalg.norm(
                np.array([_safe_ps(rx.position[0]),
                          _safe_ps(rx.position[1]),
                          _safe_ps(rx.position[2])]) - tx_pos))
            summary_rows.append({
                'receiver': rx.name,
                'x_m': _safe_ps(rx.position[0]),
                'y_m': _safe_ps(rx.position[1]),
                'z_m': _safe_ps(rx.position[2]),
                'dist_from_tx_m': los_d,
                'num_samples_used': n_samp,
                'n_paths': 0,
                'rssi_best_dbm': np.nan,
                'rssi_incoherent_dbm': np.nan,
                'rssi_coherent_dbm': np.nan,
            })
        if paths is None:
            errors += len(batch)
        if paths is not None:
            del paths
        gc.collect()
        done = min(b_start + BATCH_SIZE, total)
        if done % max(BATCH_SIZE, total // 10) < BATCH_SIZE or done == total:
            print(f'  [{done}/{total}]  {time.time()-t0:.0f}s  (0 paths — NLOS/far)')
        continue

    n_b, n_p  = a_all.shape
    tau_all   = extract_tau(paths, n_b, n_p)

    for i, rx in enumerate(batch):
        rx_pos   = np.array([_safe_ps(rx.position[0]),
                             _safe_ps(rx.position[1]),
                             _safe_ps(rx.position[2])])
        los_d    = float(np.linalg.norm(rx_pos - tx_pos))
        idx      = i if i < n_b else n_b - 1
        best_pl, incoh_pl, coh_pl, n_valid = summary_metrics(a_all[idx])

        # Convert path gain to RSSI using rssi_from_path_gain()
        _rssi_best  = rssi_from_path_gain(10**(-best_pl /10)) if not np.isnan(best_pl)  else np.nan
        _rssi_incoh = rssi_from_path_gain(10**(-incoh_pl/10)) if not np.isnan(incoh_pl) else np.nan
        _rssi_coh   = rssi_from_path_gain(10**(-coh_pl  /10)) if not np.isnan(coh_pl)   else np.nan

        summary_rows.append({
            'receiver'            : rx.name,
            'x_m'                 : _safe_ps(rx.position[0]),
            'y_m'                 : _safe_ps(rx.position[1]),
            'z_m'                 : _safe_ps(rx.position[2]),
            'dist_from_tx_m'      : los_d,
            'num_samples_used'    : n_samp,
            'n_paths'             : n_valid,
            'rssi_best_dbm'       : _rssi_best,
            'rssi_incoherent_dbm' : _rssi_incoh,
            'rssi_coherent_dbm'   : _rssi_coh,
        })

        if SAVE_PER_RAY and n_valid > 0:
            a_row   = a_all[idx]
            pwr_row = np.abs(a_row) ** 2
            order   = np.argsort(pwr_row)[::-1]
            max_pwr = pwr_row[order[0]]
            strong_phase = np.angle(a_row[order[0]], deg=True)
            for rank, ray_i in enumerate(order[:MAX_RAYS_PER_RX]):
                ac      = a_row[ray_i]
                pwr_ray = float(pwr_row[ray_i])
                if pwr_ray <= 1e-30:
                    break
                phase   = float(np.angle(ac, deg=True))
                ph_diff = (phase - strong_phase + 180) % 360 - 180
                delay   = float(tau_all[idx, ray_i]) if idx < tau_all.shape[0] and ray_i < tau_all.shape[1] and not np.isnan(tau_all[idx, ray_i]) else np.nan
                plen    = delay * _C_local if not np.isnan(delay) else np.nan
                rtype   = ray_type_heuristic(plen, los_d, pwr_ray, max_pwr) \
                          if not np.isnan(plen) else 'UNKNOWN'
                per_ray_rows.append({
                    'receiver'       : rx.name,
                    'rank'           : rank,
                    'ray_type'       : rtype,
                    'power_linear'   : pwr_ray,
                    'path_loss_db'   : -10 * np.log10(pwr_ray),
                    'amplitude_real' : float(ac.real),
                    'amplitude_imag' : float(ac.imag),
                    'phase_deg'      : phase,
                    'phase_diff_deg' : ph_diff,
                    'constructive'   : 'STRONGEST' if rank == 0
                                       else ('CONSTRUCTIVE' if abs(ph_diff) < 90 else 'DESTRUCTIVE'),
                    'delay_s'        : delay,
                    'path_length_m'  : plen,
                })

    del paths, a_all, tau_all
    gc.collect()
    done = min(b_start + BATCH_SIZE, total)
    if done % max(BATCH_SIZE, total // 10) < BATCH_SIZE or done == total:
        elapsed = time.time() - t0
        eta     = (total - done) / max(done / max(elapsed, 1e-9), 1e-9)
        print(f'  [{done}/{total}]  {elapsed:.0f}s elapsed  ETA {eta/60:.1f} min', flush=True)

# Restore all receivers
for nm in list(scene.receivers.keys()): scene.remove(nm)
for rx in _all_rx: scene.add(rx)

# ── Save ──────────────────────────────────────────────────────────────────────
df_ps = pd.DataFrame(summary_rows)
df_ps.to_csv(summary_csv, index=False)
print(f'\n  Summary  -> {summary_csv}')

if SAVE_PER_RAY and per_ray_rows:
    df_ray = pd.DataFrame(per_ray_rows)
    df_ray.to_csv(per_ray_csv, index=False)
    print(f'  Per-ray  -> {per_ray_csv}  ({len(df_ray):,} rays)')

elapsed = time.time() - t0
valid   = df_ps[df_ps['n_paths'] > 0]
nan_rx  = df_ps[df_ps['n_paths'] == 0]
print(f'\n  Total time      : {elapsed:.1f}s  |  Errors: {errors}')
print(f'  Receivers solved: {len(valid)}/{total} ({100*len(valid)/max(total,1):.1f}%)')
print(f'  Zero-path (NaN) : {len(nan_rx)}')

for col, lbl in [('rssi_incoherent_dbm', 'RSSI incoher.'),
                 ('rssi_best_dbm',        'RSSI best')]:
    v = df_ps[col].dropna()
    if len(v):
        print(f'  {lbl:20s}: mean={v.mean():.1f}  std={v.std():.1f}  min={v.min():.1f}  max={v.max():.1f}')

import matplotlib.pyplot as plt
_df_plot = df_ps.copy()
_df_plot['dist_km'] = _df_plot['dist_from_tx_m'] / 1000
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
_v = _df_plot.dropna(subset=['rssi_incoherent_dbm'])
_pl_v = [-rssi_from_path_gain(0) + r + TX_CONDUCTED_DBM + RX_EXTRA_GAIN_DB
         if not np.isnan(r) else np.nan
         for r in _v['rssi_incoherent_dbm']]
# Compute path loss from RSSI: PL = TX_CONDUCTED - RSSI + RX_EXTRA + SITE_CORR
_pl_vals = TX_CONDUCTED_DBM - _v['rssi_incoherent_dbm'] + RX_EXTRA_GAIN_DB + SITE_CORRECTION_DB
axes[0].scatter(_v['dist_km'], _pl_vals, s=5, alpha=0.5, c='steelblue')
axes[0].set(xlabel='Distance (km)', ylabel='Path Loss (dB)', title='Incoherent PL vs Distance')
axes[0].grid(alpha=0.3)
axes[1].hist(_df_plot['rssi_incoherent_dbm'].dropna(), bins=40, color='coral', edgecolor='white', alpha=0.8)
axes[1].set(xlabel='RSSI (dBm)', ylabel='Count', title='RSSI Distribution')
axes[1].grid(alpha=0.3)
axes[2].scatter(_df_plot['dist_km'], _df_plot['n_paths'], s=5, alpha=0.4, c='seagreen')
axes[2].set(xlabel='Distance (km)', ylabel='Num paths', title='Paths vs Distance')
axes[2].set_yscale('symlog')
axes[2].grid(alpha=0.3)
plt.suptitle(f'Path Solver 900 MHz (Sionna 2.0) — {len(df_ps)} receivers', fontsize=12)
plt.tight_layout()
_p = os.path.join(OUT_DIR, 'cell7_path_solver_900mhz_s2.png')
plt.savefig(_p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved → {_p}')

## Cell 8 — Compare vs Measurements

In [ ]:
# ====================================================================
# CELL 8 — SIM vs MEASUREMENTS  [900 MHz / Sionna 2.0]
# ====================================================================
import math, numpy as np, pandas as pd, matplotlib.pyplot as plt, os, glob

if not MEASUREMENT_CSV or not os.path.exists(MEASUREMENT_CSV):
    print('Set MEASUREMENT_CSV in Cell 1 to compare vs measurements.')
else:
    if 'df_ps' not in dir():
        _files = sorted(glob.glob(os.path.join(OUT_DIR, 'path_solver_summary_900s2_*.csv')))
        assert _files, f'No path solver CSV found in {OUT_DIR}\nRun Cell 7 first.'
        df_ps = pd.read_csv(_files[-1])
        print(f'Loaded: {_files[-1]}')

    df_sim = df_ps.rename(columns={
        'receiver'           : 'name',
        'rssi_incoherent_dbm': 'rssi_sim_dbm',
        'dist_from_tx_m'     : 'dist_m',
    })

    df_meas  = pd.read_csv(MEASUREMENT_CSV)
    df_merge = df_sim.merge(df_meas[['name', 'local_measurement_dBm']], on='name', how='inner')
    df_merge = df_merge.dropna(subset=['rssi_sim_dbm', 'local_measurement_dBm'])
    df_merge['err']     = df_merge['rssi_sim_dbm'] - df_merge['local_measurement_dBm']
    df_merge['dist_km'] = df_merge['dist_m'] / 1000

    bias = df_merge['err'].mean()
    rmse = math.sqrt((df_merge['err']**2).mean())

    print(f'Receivers compared : {len(df_merge)}')
    print(f'Bias (sim-meas)    : {bias:+.2f} dB')
    print(f'RMSE               : {rmse:.2f} dB')
    print()

    print(f'  {"Band":<12} {"N":>4}  {"Bias (dB)":>10}  {"RMSE (dB)":>10}  {"Mean paths":>11}')
    print(f'  {"-"*12} {"-"*4}  {"-"*10}  {"-"*10}  {"-"*11}')
    for lbl, d0, d1 in [("<300m",0,300),("300-700m",300,700),
                        ("700m-1.2km",700,1200),(">1.2km",1200,9999)]:
        s = df_merge[(df_merge['dist_m']>=d0) & (df_merge['dist_m']<d1)]
        if not len(s): continue
        e = s['err']
        p = s['n_paths'].mean() if 'n_paths' in s.columns else float('nan')
        print(f'  {lbl:<12} {len(s):>4}  {e.mean():>+10.1f}  {((e**2).mean()**0.5):>10.1f}  {p:>11.0f}')

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].scatter(df_merge['dist_km'], df_merge['err'], s=6, alpha=0.5, color='steelblue')
    axes[0].axhline(0, color='red', lw=1)
    axes[0].set_xlabel('Distance (km)')
    axes[0].set_ylabel('Error (dB)')
    axes[0].set_title(f'Error vs distance  (bias={bias:+.1f} dB, RMSE={rmse:.1f} dB)')
    axes[0].grid(alpha=0.3)

    axes[1].scatter(df_merge['local_measurement_dBm'], df_merge['rssi_sim_dbm'],
                    s=6, alpha=0.5, color='steelblue')
    _lo = min(df_merge['local_measurement_dBm'].min(), df_merge['rssi_sim_dbm'].min()) - 5
    _hi = max(df_merge['local_measurement_dBm'].max(), df_merge['rssi_sim_dbm'].max()) + 5
    axes[1].plot([_lo, _hi], [_lo, _hi], 'r--', lw=1)
    axes[1].set_xlabel('Measured RSSI (dBm)')
    axes[1].set_ylabel('Simulated RSSI (dBm)')
    axes[1].set_title('Sim vs Measured  (900 MHz / Sionna 2.0)')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    _p = os.path.join(OUT_DIR, 'rssi_compare_900mhz_s2.png')
    plt.savefig(_p, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot saved → {_p}')